# M1 — Laboratoire LoRA DiagOps

Ce notebook guide l'execution. Avant chaque run, completez `templates/protocol_m1.md` et la configuration associee. Le test final reste ferme jusqu'au Brief 2.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

STARTER = Path.cwd().resolve()
if STARTER.name == 'notebooks':
    STARTER = STARTER.parent
print('Starter:', STARTER)

## 1. Relever l'environnement

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.environment',
    '--output', 'work/environment.json'
], cwd=STARTER, check=True)
json.loads((STARTER / 'work/environment.json').read_text())

## 2. Creer le split 320 / 80

Cette etape utilise uniquement `diagops_train.jsonl`.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.dataset',
    '--input', '../../data_pack/2026-S1/annotations/diagops_train.jsonl',
    '--output-dir', 'work/splits',
    '--seed', '42',
    '--validation-size', '80'
], cwd=STARTER, check=True)
json.loads((STARTER / 'work/splits/split_manifest.json').read_text())

## 3. Inspecter un exemple

Ne corrigez pas silencieusement une annotation. Notez toute anomalie dans l'analyse.

In [ ]:
with (STARTER / 'work/splits/train.jsonl').open() as handle:
    example = json.loads(next(handle))
example

## 4. Verifier la revision gelee

Le modele est epingle sur la revision Hugging Face `c1899de289a04d12100db370d81485cdf75e47ca`.

In [ ]:
baseline_config = (STARTER / 'configs/baseline.yaml').read_text()
PINNED_REVISION = 'c1899de289a04d12100db370d81485cdf75e47ca'
assert PINNED_REVISION in baseline_config, 'Revision epinglee absente'
print(f'Revision configuree : {PINNED_REVISION}')

## 5. Baseline sur la validation

Executez cette cellule apres avoir gele le protocole.

In [ ]:
# Decommenter pour executer sur l'environnement commun.
# subprocess.run([
#     sys.executable, '-m', 'src.evaluate',
#     '--config', 'configs/baseline.yaml',
#     '--data', 'work/splits/validation.jsonl',
#     '--output-dir', 'work/baseline_validation'
# ], cwd=STARTER, check=True)

## 6. Run LoRA de reference

In [ ]:
# Decommenter pour executer sur l'environnement CUDA commun.
# subprocess.run([
#     sys.executable, '-m', 'src.train',
#     '--config', 'configs/lora_reference.yaml',
#     '--train-data', 'work/splits/train.jsonl',
#     '--output-dir', 'work/runs/lora_reference'
# ], cwd=STARTER, check=True)

## 7. Evaluer le LoRA sur la validation

In [ ]:
# Decommenter apres le run.
# subprocess.run([
#     sys.executable, '-m', 'src.evaluate',
#     '--config', 'configs/baseline.yaml',
#     '--adapter', 'work/runs/lora_reference/adapter',
#     '--data', 'work/splits/validation.jsonl',
#     '--output-dir', 'work/lora_reference_validation'
# ], cwd=STARTER, check=True)

## 8. Comparer et documenter

Copiez `configs/variation_template.yaml` pour les deux variations. Completez ensuite les gabarits `error_analysis.md` et `peer_review.md`. Le jeu de test final n'est pas utilise dans ce notebook pendant le Brief 1.